# Study 02: Decoder Artifact Debugging
**Goal:** Detect checkerboard artifacts in the upsampling path and compare bilinear vs PixelShuffle strategies.

## 1. Setup

In [ ]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from src.utils import setup_logging, logger
from src.config import DEVICE
from src.models import create_model
warnings.filterwarnings('ignore')
print(f'Device: {DEVICE}, Torch: {torch.__version__}')

## 2. Load Model & Checkpoint

In [ ]:
model = create_model('mobilenetv2_unet', in_channels=1, out_channels=1)
ckpt = torch.load(Path.cwd().parent/'models'/'best_model.pth', map_location='cpu', weights_only=True)
if isinstance(ckpt, dict) and 'model_state' in ckpt: ckpt = ckpt['model_state']
model.load_state_dict(ckpt, strict=False)
model.to(DEVICE); model.eval()
print('Model loaded')

## 3. Gradient Analysis (Detect Anisotropic Artifacts)

In [ ]:
# Extract decoder conv weights and compute gradient anisotropy
anisotropy = {}
for name, param in model.named_parameters():
    if 'conv' in name and param.dim() == 4 and 'decoder' in name:
        w = param.data.cpu().numpy()
        gx = np.abs(np.diff(w.mean(axis=(0,1)), axis=1)).mean()
        gy = np.abs(np.diff(w.mean(axis=(0,1)), axis=0)).mean()
        ratio = gx / max(gy, 1e-8)
        anisotropy[name] = {'gx': float(gx), 'gy': float(gy), 'ratio': float(ratio)}

# Plot anisotropy
names = list(anisotropy.keys())
ratios = [anisotropy[n]['ratio'] for n in names]
fig, ax = plt.subplots(figsize=(12, 4))
colors = ['red' if r < 0.8 or r > 1.2 else 'green' for r in ratios]
ax.bar(range(len(names)), ratios, color=colors)
ax.axhline(1.0, color='gray', ls='--', label='Ideal (isotropic)')
ax.axhline(0.8, color='orange', ls=':', label='Warning threshold')
ax.axhline(1.2, color='orange', ls=':')
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.split('.')[-1] for n in names], rotation=45, ha='right')
ax.set_ylabel('Gradient anisotropy (gx/gy)')
ax.set_title('Decoder Conv — Gradient Anisotropy')
ax.legend()
plt.tight_layout()
plt.savefig(Path.cwd().parent/'figures'/'gradient_anisotropy.png', dpi=150)
plt.show()

## 4. FFT Analysis (Checkerboard Frequency Detection)

In [ ]:
def fft_analysis(tensor, title):
    t = tensor.detach().cpu().numpy()
    if t.ndim == 4: t = t.mean(axis=(0,1))
    f = np.fft.fft2(t)
    fshift = np.fft.fftshift(f)
    magnitude = np.log(np.abs(fshift) + 1)
    h, w = magnitude.shape
    cy, cx = h//2, w//2
    center = magnitude[cy-3:cy+4, cx-3:cx+4].mean()
    ring = magnitude[cy-10:cy+11, cx-10:cx+11].mean()
    ratio = ring / max(center, 1e-8)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].imshow(t, cmap='gray'); axes[0].set_title(f'{title} (spatial)')
    im = axes[1].imshow(magnitude, cmap='inferno'); axes[1].set_title('FFT magnitude')
    axes[2].imshow(magnitude[cy-20:cy+21, cx-20:cx+21], cmap='inferno')
    axes[2].set_title(f'FFT center (ring/center={ratio:.2f})')
    for ax in axes: ax.axis('off')
    plt.tight_layout(); plt.show()
    return ratio

# Test on random input
x = torch.randn(1, 1, 256, 256, device=DEVICE)
with torch.no_grad():
    f0 = model.enc_0(x)
    print(f'FFT ratio for enc_0 output:')
    r = fft_analysis(f0[0,0], 'encoder output')

## 5. PixelShuffle Comparison

In [ ]:
class PixelShuffleHead(nn.Module):
    def __init__(self, in_ch=16, out_ch=1, up_factor=2):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch * up_factor * up_factor, kernel_size=3, padding=1)
        self.shuffle = nn.PixelShuffle(up_factor)
    def forward(self, x): return self.shuffle(self.conv(x))

# Build alternate model with PixelShuffle
alt_model = create_model('mobilenetv2_unet', in_channels=1, out_channels=1)
alt_model.load_state_dict(ckpt, strict=False)
def _alt_forward(self, x):
    f0=self.enc_0(x); f1=self.enc_1(f0); f2=self.enc_2(f1)
    f3=self.enc_3(f2); f4=self.enc_4(f3); f5=self.enc_5(f4)
    dec=self.decoder([f0,f1,f2,f3,f4,f5])
    return self.final(dec)
alt_model.forward = _alt_forward.__get__(alt_model)
alt_model.final = PixelShuffleHead(16, 1, 2)
alt_model.to(DEVICE); alt_model.eval()
print('Alternate model ready')

## 6. Visual Comparison: Bilinear vs PixelShuffle

In [ ]:
candidates = list(Path('D:/DATA SCIENCE AND ANALYTICS/Dataset/Liver Img Dataset').glob('Volume-001-*.png'))
img_pil = Image.open(str(candidates[0])).convert('L')
img = np.array(img_pil, dtype=np.float32) / 255.0
img_t = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    out_base = torch.sigmoid(model(img_t)).cpu().numpy()[0,0]
    out_alt = torch.sigmoid(alt_model(img_t)).cpu().numpy()[0,0]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(out_base, cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'Bilinear + 1x1 Conv\n({out_base.shape[0]}\u00d7{out_base.shape[1]})')
axes[0].axis('off')
axes[1].imshow(out_alt, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'PixelShuffle Head\n({out_alt.shape[0]}\u00d7{out_alt.shape[1]})')
axes[1].axis('off')
diff = axes[2].imshow(np.abs(out_base - out_alt), cmap='hot', vmin=0, vmax=0.1)
axes[2].set_title('|Difference| (hot = large change)')
axes[2].axis('off')
plt.colorbar(diff, ax=axes[2], shrink=0.8)
plt.tight_layout()
plt.savefig(Path.cwd().parent/'figures'/'head_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Checkerboard Artifact Score

In [ ]:
# Compute grid artifact score: look for periodic patterns at Nyquist frequency
def checkerboard_score(img):
    h, w = img.shape
    # Difference of shifted images reveals checkerboard
    dx = np.abs(img[:,1:] - img[:,:-1]).mean()
    dy = np.abs(img[1:,:] - img[:-1,:]).mean()
    return float(dx + dy)

score_base = checkerboard_score(out_base)
score_alt = checkerboard_score(out_alt)
print(f'Checkerboard score (bilinear): {score_base:.6f}')
print(f'Checkerboard score (pixel_shuffle): {score_alt:.6f}')
print(f'Improvement: {(1-score_alt/score_base)*100:.1f}%' if score_alt < score_base else 'No improvement')

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Bilinear', 'PixelShuffle'], [score_base, score_alt], color=['orange', 'teal'])
ax.set_ylabel('Checkerboard artifact score')
ax.set_title('Artifact Score Comparison\n(lower = better)')
plt.tight_layout(); plt.show()

## 8. Summary

In [ ]:
print('='*60)
print('FINDINGS')
print('='*60)
aniso_flags = [(n, a['ratio']) for n, a in anisotropy.items() if a['ratio'] < 0.8 or a['ratio'] > 1.2]
print(f'Anisotropic layers: {len(aniso_flags)}/{len(anisotropy)}')
for n, r in aniso_flags:
    print(f'  {n.split(".")[-1]}: ratio={r:.3f}')
print(f'\nPixelShuffle reduces artifacts: {score_alt < score_base}')
print(f'Recommendation: Replace F.interpolate with PixelShuffle in forward()')